# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsf-rawnak/FlyRankAI-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Data lineage for this run:** this notebook rebuilds the Week 5/6 method inline — **Logistic
Regression, features and split exactly as `w05_model.ipynb`, client-grouped `GroupShuffleSplit`
(seed 42)** — rather than importing those notebooks directly, so this one stays independently
runnable top to bottom. The held-out result it reproduces is Week 5/6's own number:
**Precision@50 = 0.86** vs the Week-4 rule baseline's **0.34**, confirmed honest (not a random-split
artifact) in `w06_validation_audit.ipynb`'s before/after check (0.92 naive → 0.86 grouped) and its
leakage-injection test (adding `trend_pct` back pushes the score to 1.00, confirming the harness
would have caught it if it were really there).

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**The rule in plain words:** refit the validated Week-5 Logistic Regression on the *full* catalog
(after confirming its honest generalization number on the held-out client split below), score
every item with it, then turn that probability plus a few transparent flags into a reason code and
one of four plain actions:

| Action | What it means | Relative human cost |
|---|---|---|
| `monitor` | leave alone, just keep watching | lowest — no work needed |
| `refresh` | update the content (facts, links, structure) | medium — a rewrite pass |
| `refresh_and_review_ctr` | refresh **and** check title/meta for the CTR gap | medium-high |
| `technical_check_unranked` | not a content problem — a visibility/indexing problem | its own lane, see below |

**Why a fourth action, and why it isn't a probability threshold:** the model assigns
**near-zero decline probability to every unranked page** (`avg_position == 0`, mean predicted
probability ≈ 0.007) — not because those pages are healthy, but because a page with no recorded
position has nothing for the model's signals to read as "declining." That's a real blind spot,
not a reassuring score, so unranked pages get pulled into their own action lane by rule, before
the model probability is even consulted — see Section 2 for why this matters as a limit.

**Cost/value thinking:** review time is the scarce resource. High-confidence items do carry the
most traffic per item on average — but the ordering isn't perfectly clean below medium and low
confidence, which is itself worth knowing: "confidence" here reflects the model's certainty about
the *label*, not a direct measure of business value. A reviewer should still glance at impressions
before skipping a "low confidence" row.

**Archetype → action mapping:** `content_type × main_intent`, descriptive of this sample only.
`keyword article / navigational` (n=46) is too thin to trust a mapping for — flagged, not shown as
a rate.

**Decay/refresh insight:** refresh-family action rate is **not monotonic with content age** — it
peaks for 91–180 day content (~70%) and is *lowest* for the oldest tier, 365+ days (~28%). Read
plainly: the oldest surviving content in this catalog looks the most stable, not the most at risk
— age alone is a weak, non-monotonic signal; the model's own decline signal is doing the real
work here, not "this page is old."

In [11]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

local_path = Path("../../data/raw/content_refresh_anonymized.csv")
raw_url = "https://raw.githubusercontent.com/rsf-rawnak/FlyRankAI-ML-Internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(local_path) if local_path.exists() else pd.read_csv(raw_url)

# --- rebuild Week 5's leak-safe feature set exactly ---
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
base_rate = df["is_declining_label"].mean()

df["has_search_volume"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_avg_position"] = (df["avg_position"] > 0).astype(int)  # avg_position==0 means "no data"
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
for c in ["search_volume", "competition", "cpc", "word_count", "char_count", "scroll_rate"]:
    df[c] = df[c].fillna(df[c].median())
df["avg_position_filled"] = df["avg_position"].replace(0, 100)

num_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "content_age_days", "days_since_last_update", "ctr", "avg_position_filled",
    "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "has_search_volume", "has_word_count", "has_avg_position",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d",
]
cat_features = ["content_type", "main_intent", "competition_level", "age_tier", "freshness_tier"]

leaked_cols = {"trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
               "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d", "is_declining_label"}
assert not (set(num_features + cat_features) & leaked_cols), "leak check failed"

X = df[num_features + cat_features].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].copy()

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
])

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

K = 50

# --- honest check: same client-grouped split as Week 5/6 (this is the VALIDATION step, not deployment) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
model_holdout = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000, random_state=42))])
model_holdout.fit(X.iloc[train_idx], y.iloc[train_idx])
model_p50 = precision_at_k(model_holdout.predict_proba(X.iloc[test_idx])[:, 1], y.iloc[test_idx], K)

dft = df.iloc[test_idx]
stale_flag = (dft["days_since_last_update"] >= 90).astype(int)
ctr_weak_flag = ((dft["avg_position"] > 0) & (dft["avg_position"] <= 20) & (dft["ctr"] < 0.30)).astype(int)
visible_flag = (dft["impressions_90d"] >= 250).astype(int)
baseline_scores = visible_flag * (1 + stale_flag + ctr_weak_flag) * np.log1p(dft["impressions_90d"])
baseline_p50 = precision_at_k(baseline_scores, y.iloc[test_idx], K)

print(f"Held-out precision@{K} — Logistic Regression: {model_p50:.3f} | Week-4 rule baseline: {baseline_p50:.3f}")
print(f"Lift: {model_p50/baseline_p50:.2f}x | client overlap train/test (must be 0): "
      f"{len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))}")

# --- deployment scoring: refit on the FULL catalog to score everything for the playbook ---
# (validated above on a held-out split; this refit is standard practice once the method is
# trusted — it is NOT what the 0.86 above measures, and the notebook is explicit about that.)
model_full = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000, random_state=42))])
model_full.fit(X, y)
df["model_probability"] = model_full.predict_proba(X)[:, 1]
print(f"\nFull-catalog scoring done. Base rate: {base_rate:.3f} | "
      f"mean predicted probability: {df['model_probability'].mean():.3f}")

Held-out precision@50 — Logistic Regression: 0.860 | Week-4 rule baseline: 0.340
Lift: 2.53x | client overlap train/test (must be 0): 0

Full-catalog scoring done. Base rate: 0.542 | mean predicted probability: 0.542


In [12]:
# --- reason-code flags + suggested action ---
stale = df["days_since_last_update"] >= 90
ctr_weak = (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.30)
visible = df["impressions_90d"] >= 250
unranked = df["has_avg_position"] == 0
decline_risk = df["model_probability"] >= 0.65
opportunity = (df["model_probability"] >= 0.50) & visible

def build_reason_codes(row):
    reasons = []
    if row["unranked"]:
        reasons.append("no_indexed_position")
    if row["decline_risk"]:
        reasons.append("model_decline_risk")
    if row["opportunity"]:
        reasons.append("visible_model_opportunity")
    if row["ctr_weak"]:
        reasons.append("low_ctr_visible_page")
    if row["stale"]:
        reasons.append("stale_content")
    return "|".join(reasons) if reasons else "general_monitor"

def suggested_action(row):
    if row["unranked"]:
        return "technical_check_unranked"
    if row["ctr_weak"] and (row["decline_risk"] or row["opportunity"]):
        return "refresh_and_review_ctr"
    if row["decline_risk"] or (row["stale"] and row["opportunity"]) or row["opportunity"]:
        return "refresh"
    return "monitor"

flags = pd.DataFrame({"unranked": unranked, "decline_risk": decline_risk, "ctr_weak": ctr_weak,
                       "stale": stale, "opportunity": opportunity})
df["reason_codes"] = flags.apply(build_reason_codes, axis=1)
df["suggested_action"] = flags.apply(suggested_action, axis=1)

sessions_ok = df["sessions_90d"] >= 10
df["confidence"] = np.select(
    [(df["suggested_action"] != "monitor") & visible & sessions_ok & (df["model_probability"] >= 0.65),
     (df["suggested_action"] != "monitor")],
    ["high", "medium"], default="low",
)

action_counts = df["suggested_action"].value_counts()
print("Action mix across the full catalog (n =", len(df), "):")
print((action_counts.astype(str) + "  (" + (100 * action_counts / len(df)).round(1).astype(str) + "%)").to_string())

ranked_actions = df.sort_values("model_probability", ascending=False)[
    ["content_id", "model_probability", "confidence", "suggested_action", "reason_codes",
     "content_type", "main_intent", "impressions_90d", "sessions_90d", "trend_direction"]
]
print("\nTop of the queue (by model probability, excluding the technical-check lane which sorts separately):")
display(ranked_actions[ranked_actions["suggested_action"] != "technical_check_unranked"].head(10))

Action mix across the full catalog (n = 30000 ):
suggested_action
monitor                     12519  (41.7%)
refresh_and_review_ctr       8889  (29.6%)
refresh                      7387  (24.6%)
technical_check_unranked      1205  (4.0%)

Top of the queue (by model probability, excluding the technical-check lane which sorts separately):


,content_id,model_probability,confidence,suggested_action,reason_codes,content_type,main_intent,impressions_90d,sessions_90d,trend_direction
23220,content_f986bd514b6e,0.944438,medium,refresh_and_review_ctr,model_decline_risk|visible_model_opportunity|l...,keyword article,NaN,22456,4,down
7445,content_c8e9d6ab9013,0.939707,medium,refresh_and_review_ctr,model_decline_risk|visible_model_opportunity|l...,keyword article,informational,208678,6,down
9629,content_c90bfc85694f,0.932188,medium,refresh_and_review_ctr,model_decline_risk|visible_model_opportunity|l...,keyword article,commercial,3047,1,down
17362,content_c82bc0c24241,0.932149,medium,refresh_and_review_ctr,model_decline_risk|visible_model_opportunity|l...,keyword article,informational,13676,3,down
4533,content_d6e1bbb4a996,0.922510,medium,refresh_and_review_ctr,model_decline_risk|visible_model_opportunity|l...,keyword article,informational,4955,1,down
9443,content_8ba781dafa55,0.919061,medium,refresh_and_review_ctr,model_decline_risk|visible_model_opportunity|l...,keyword article,informational,16156,2,down
24866,content_e5f459e737b7,0.915482,high,refresh_and_review_ctr,model_decline_risk|visible_model_opportunity|l...,keyword article,transactional,56363,17,down
2646,content_87c007fb5c26,0.915217,medium,refresh_and_review_ctr,model_decline_risk|visible_model_opportunity|l...,keyword article,informational,2463,2,down
24076,content_4bb993e9270e,0.914379,medium,refresh_and_review_ctr,model_decline_risk|visible_model_opportunity|l...,keyword article,informational,1986,2,down
18587,content_823ea9b9b355,0.914195,medium,refresh_and_review_ctr,model_decline_risk|visible_model_opportunity|l...,keyword article,informational,4369,1,down


In [13]:
# --- cost/value check ---
value_by_confidence = (
    df.groupby("confidence")
    .agg(n=("content_id", "size"), total_impressions_90d=("impressions_90d", "sum"))
    .reindex(["high", "medium", "low"])
)
value_by_confidence["impressions_per_item"] = (
    value_by_confidence["total_impressions_90d"] / value_by_confidence["n"]
).round(0)
print("Traffic at stake per item, by confidence tier (ordering is NOT perfectly clean — see note above):")
display(value_by_confidence)

Traffic at stake per item, by confidence tier (ordering is NOT perfectly clean — see note above):


,n,total_impressions_90d,impressions_per_item
confidence,,,
high,3957,28083371,7097.0
medium,13524,61199375,4525.0
low,12519,66728243,5330.0


In [14]:
# --- archetype -> action mapping (content_type x main_intent), descriptive only ---
df["archetype"] = df["content_type"] + " / " + df["main_intent"].fillna("unknown")
archetype_counts = df["archetype"].value_counts()
kept_archetypes = archetype_counts[archetype_counts >= 200].index
small_archetypes = archetype_counts[archetype_counts < 200]

archetype_map = pd.crosstab(
    df.loc[df["archetype"].isin(kept_archetypes), "archetype"],
    df.loc[df["archetype"].isin(kept_archetypes), "suggested_action"],
    normalize="index",
).round(3)
archetype_map["n"] = archetype_counts.loc[kept_archetypes]
archetype_map["avg_model_probability"] = (
    df[df["archetype"].isin(kept_archetypes)].groupby("archetype")["model_probability"].mean().round(3)
)
print("Archetype -> action share (archetypes with n >= 200 only):")
display(archetype_map)
print("\nArchetypes too thin to trust a mapping for (n < 200) — human judgement only:")
print(small_archetypes.to_string())

Archetype -> action share (archetypes with n >= 200 only):


suggested_action,monitor,refresh,refresh_and_review_ctr,technical_check_unranked,n,avg_model_probability
archetype,,,,,,
comparison article / informational,0.690,0.059,0.251,0.000,697,0.575
feedly article / unknown,0.542,0.029,0.081,0.348,2096,0.286
keyword article / commercial,0.410,0.261,0.312,0.017,4612,0.551
keyword article / informational,0.386,0.280,0.317,0.016,16538,0.570
keyword article / transactional,0.431,0.242,0.308,0.019,5733,0.543
keyword article / unknown,0.417,0.194,0.313,0.076,278,0.567



Archetypes too thin to trust a mapping for (n < 200) — human judgement only:
archetype
keyword article / navigational    46


In [15]:
# --- decay/refresh insight: is "older" the same as "needs a refresh"? ---
df["refresh_family"] = df["suggested_action"].isin(["refresh", "refresh_and_review_ctr"])
age_order = ["31-90", "91-180", "181-365", "365+"]
decay_view = (
    df.groupby("age_tier")
    .agg(n=("content_id", "size"),
         refresh_family_rate=("refresh_family", "mean"),
         declining_label_rate=("is_declining_label", "mean"),
         avg_days_since_last_update=("days_since_last_update", "mean"))
    .reindex(age_order)
    .round(3)
)
print("Refresh-family action rate by content age tier — NOT monotonic:")
display(decay_view)
print("\nObserved, not causal: the oldest tier (365+) has the LOWEST refresh-family rate, not the")
print("highest — the surviving old content looks stable, not at-risk, in this dataset.")

Refresh-family action rate by content age tier — NOT monotonic:


,n,refresh_family_rate,declining_label_rate,avg_days_since_last_update
age_tier,,,,
31-90,492,0.713,0.669,17.848
91-180,11780,0.704,0.626,33.907
181-365,11368,0.517,0.515,70.468
365+,6360,0.275,0.426,27.305



Observed, not causal: the oldest tier (365+) has the LOWEST refresh-family rate, not the
highest — the surviving old content looks stable, not at-risk, in this dataset.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use.** This queue is a **decision-support ranking for a human content reviewer**,
scoped to FlyRank's internal editorial/SEO team. It answers "which pages should I look at first,
this week, and why" — nothing more. On the held-out client split (client-grouped, matching Week
5/6), Logistic Regression ranks true decliners into the top 50 at **precision@50 = 0.86** against
a **54.2%** base rate, roughly a **2.5×** lift over the transparent hand-rule baseline's **0.34**.
`w06_validation_audit.ipynb` confirmed this isn't a naive-split artifact (a random row-level split
inflated the same model to 0.92 with 31 clients leaking across the split) and confirmed the test
harness itself catches leakage when it's deliberately introduced (adding `trend_pct` back pushes
the score to 1.00). That is a measured, validated ranking improvement — it is not a claim about
*why* a page is declining, and it is not a claim that acting on any single row will improve it.

**Limits, stated plainly:**

- **Cross-sectional, one snapshot.** Nothing here supports "refreshing this page will increase
  traffic" — only "this page looks worth reviewing first, because…". That distinction matters in
  every sentence written from this queue.
- **One catalog, one period.** Precision@50 = 0.86 describes *this* 30k-row anonymized sample
  under *this* client-holdout split. It is not a guarantee that a different client mix, a
  different quarter, or a different content type distribution behaves the same way.
- **The model is functionally blind to unranked pages.** `avg_position == 0` rows (1,205 of them,
  4.0% of the catalog) get a mean predicted probability of **0.007** — near-zero across the
  board, every time, regardless of anything else about the page. This is not the model saying
  "these are healthy"; it is the model having nothing to read a decline signal from. Section 5's
  Week-5 error analysis found exactly this case as a false negative (an unranked page that *did*
  decline, scored as low-risk). These pages are routed to a separate, non-model action lane —
  never filtered out by probability.
- **Uneven archetype coverage.** `keyword article` is ~90% of the catalog; `keyword article /
  navigational` (n=46) is too thin for the archetype map to be trusted at the same confidence
  level as the larger groups.
- **Confidence tiers are a review triage, not a guarantee.** ~42% of the catalog sits in
  "low confidence" and ~18% sits in the probability band 0.45–0.55 — the model itself is saying
  it doesn't know enough there to prioritize confidently; that's a signal to look manually, not a
  wrong answer.
- **`is_declining_label` is derived from this dataset's own `trend_pct`.** It measures a pattern
  inside this data, not a certified fact about a page's real-world performance (see
  `flyrank-data` skill, "the label trap").

In [16]:
# supporting numbers for Section 2 (computed live above; restated here for the write-up)
lift = model_p50 / baseline_p50
borderline = df[(df["model_probability"] >= 0.45) & (df["model_probability"] <= 0.55)]
unranked_mean_prob = df.loc[df["has_avg_position"] == 0, "model_probability"].mean()
low_conf_share = (df["confidence"] == "low").mean()

print(f"model precision@50 (held-out): {model_p50:.3f}")
print(f"baseline precision@50 (held-out): {baseline_p50:.3f}")
print(f"lift: {lift:.2f}x")
print(f"base rate (declining label): {base_rate:.3f}")
print(f"unranked-page mean predicted probability: {unranked_mean_prob:.3f}")
print(f"borderline-probability rows (0.45-0.55): {len(borderline):,} ({len(borderline)/len(df):.1%})")
print(f"low-confidence share: {low_conf_share:.1%}")

model precision@50 (held-out): 0.860
baseline precision@50 (held-out): 0.340
lift: 2.53x
base rate (declining label): 0.542
unranked-page mean predicted probability: 0.007
borderline-probability rows (0.45-0.55): 5,431 (18.1%)
low-confidence share: 41.7%


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before anyone acts on a row, a human confirms, per item:**

1. The page still exists and matches what the queue thinks it is (pseudonymized IDs mean the
   queue can't show titles/URLs — the reviewer must look the item up in FlyRank's own systems).
2. If `suggested_action == "technical_check_unranked"`, this is **not** a content-quality call —
   it's "why isn't this indexed/ranked at all" (crawl, indexing, canonicalization, robots rules).
   Content refresh doesn't help a page nothing can find.
3. For `confidence == "low"` rows or rows in the 0.45–0.55 probability band, treat the row as
   "worth a look," not "worth an action" — the model is explicitly uncertain there.
4. For the thin `keyword article / navigational` archetype (n=46), don't apply the majority-class
   mapping automatically — too few examples for the model to have generalized confidently.

**No-go — should NOT be automated, full stop:**

- **No automatic publishing, unpublishing, or content edits.** The queue produces a *reading
  list* for a human editor, never a write action to a CMS.
- **No automatic client-facing communication** ("your page is declining") built directly from a
  row's score or reason codes.
- **No batch `technical_check_unranked` actions.** Each one needs a person to check *why* it's
  unranked before anything is "fixed" — a wrong guess here can waste real engineering time.
- **No cross-client generalization of the archetype mapping** — descriptive of *this* anonymized
  sample only.
- **No treating `is_declining_label` as ground truth about the real world** — it is derived from
  this dataset's own `trend_pct`, not an external fact (see `flyrank-data` skill).

In [17]:
# --- turn the no-go list into a check, not just prose ---
needs_human_review = (
    (df["suggested_action"] == "technical_check_unranked")
    | (df["confidence"] == "low")
    | ((df["model_probability"] >= 0.45) & (df["model_probability"] <= 0.55))
    | (df["archetype"].isin(small_archetypes.index))
)
review_share = needs_human_review.mean()
print(f"Rows the no-go rules route to mandatory human review before any action: "
      f"{needs_human_review.sum():,} of {len(df):,} ({review_share:.1%})")
print("\nBreakdown of which no-go condition fired (rows can match more than one):")
print("technical_check_unranked      :", (df["suggested_action"] == "technical_check_unranked").sum())
print("low confidence                 :", (df["confidence"] == "low").sum())
print("borderline probability         :", len(borderline))
print("thin archetype                 :", df["archetype"].isin(small_archetypes.index).sum())

Rows the no-go rules route to mandatory human review before any action: 15,663 of 30,000 (52.2%)

Breakdown of which no-go condition fired (rows can match more than one):
technical_check_unranked      : 1205
low confidence                 : 12519
borderline probability         : 5431
thin archetype                 : 46


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

This stays a **non-production, review-cadence check** — there is no live pipeline here, just a
recurring manual snapshot compared against the numbers this notebook just produced (the
"monitoring baseline," written out in Section 5 so it's a receipt, not a memory).

**Check on a recurring cadence (e.g. monthly, re-running the held-out split above fresh):**

| Signal | Current snapshot | Trigger to investigate / consider retraining |
|---|---|---|
| Held-out precision@50 | 0.86 | Drops below ~1.5× the baseline rule's precision@50 (currently 0.34), i.e. below ~0.51 — the model's whole value is beating the transparent rule by a wide, clear margin |
| Declining-label base rate | 54.2% | Moves by more than ~10 points in a quarter — since the label is derived from `trend_pct`, a shift this large usually means an upstream metric definition changed, not that content quality changed |
| Unranked-page share | 4.0% of catalog | Rises sharply — since the model can't score these at all, a growing unranked share shrinks the fraction of the catalog the model is actually useful for |
| Content-type / archetype mix | `keyword article` ≈ 90% of rows | A new `content_type` appears, or an archetype's share moves >15 points — the model has no exposure to a materially different mix |
| No-go / human-review share | ~current share of the queue routed to mandatory review | Drops sharply — check whether the no-go conditions are still being computed correctly before trusting it as "the model got better" |

**Retrain, don't just re-score, when:** the held-out precision@50 trigger fires twice in a row, or
the archetype/content-type mix has genuinely shifted — a quick re-score on stale features won't
fix either of those; the feature set and split need a fresh look, with the same client-holdout
discipline as Week 5/6, not just a re-run of this notebook's numbers.

In [18]:
# --- build the monitoring baseline snapshot: the numbers a future run gets compared against ---
conf_counts = df["confidence"].value_counts(normalize=True).reindex(["high", "medium", "low"]).fillna(0)
unranked_share = (df["suggested_action"] == "technical_check_unranked").mean()

monitoring_baseline = {
    "generated_from": "work/notebooks/w07_action_playbook.ipynb (rebuilds Week-5/6 Logistic Regression)",
    "model_name": "logistic_regression",
    "split_strategy": "client_grouped (GroupShuffleSplit, seed=42, test_size=0.20)",
    "held_out_precision_at_50_model": round(model_p50, 4),
    "held_out_precision_at_50_baseline": round(baseline_p50, 4),
    "declining_label_base_rate": round(float(base_rate), 4),
    "unranked_page_share": round(float(unranked_share), 4),
    "unranked_page_mean_probability": round(float(unranked_mean_prob), 4),
    "confidence_mix": {k: round(float(v), 4) for k, v in conf_counts.items()},
    "content_type_mix": df["content_type"].value_counts(normalize=True).round(4).to_dict(),
    "no_go_human_review_share": round(float(review_share), 4),
    "catalog_size": int(len(df)),
}
print(json.dumps(monitoring_baseline, indent=2))

{
  "generated_from": "work/notebooks/w07_action_playbook.ipynb (rebuilds Week-5/6 Logistic Regression)",
  "model_name": "logistic_regression",
  "split_strategy": "client_grouped (GroupShuffleSplit, seed=42, test_size=0.20)",
  "held_out_precision_at_50_model": 0.86,
  "held_out_precision_at_50_baseline": 0.34,
  "declining_label_base_rate": 0.5421,
  "unranked_page_share": 0.0402,
  "unranked_page_mean_probability": 0.0069,
  "confidence_mix": {
    "high": 0.1319,
    "medium": 0.4508,
    "low": 0.4173
  },
  "content_type_mix": {
    "keyword article": 0.9069,
    "feedly article": 0.0699,
    "comparison article": 0.0232
  },
  "no_go_human_review_share": 0.5221,
  "catalog_size": 30000
}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Three things get written out:

1. **`work/outputs/w07_action_playbook_queue.csv`** — the top 300 rows (ranked by model
   probability, excluding the technical-check lane) with action, reason codes, confidence, and
   archetype. This is what the paper's recommendations section pulls from. Gitignored by design
   (CI's leak-guard blocks data files) — the notebook regenerates it every run.
2. **`work/outputs/w07_unranked_technical_check.csv`** — the full `technical_check_unranked` list
   (1,205 rows), kept separate because it's a different kind of list — a technical/indexing
   check, not a content-priority ranking.
3. **`work/outputs/w07_playbook_summary.json`** — every number cited above in one place. This one
   **stays committed**: it's the receipt the paper's numbers trace back to.
4. **`work/figures/`** — action-mix and confidence-mix bar charts, generated fresh from this run.

In [19]:
import sys

WORK_OUTPUTS = Path("../outputs")
WORK_FIGURES = Path("../figures")
WORK_OUTPUTS.mkdir(parents=True, exist_ok=True)
WORK_FIGURES.mkdir(parents=True, exist_ok=True)

# 1. main ranked queue (top 300, excludes the technical-check lane) — stays out of git
export_cols = [
    "content_id", "client_id", "model_probability", "confidence", "suggested_action",
    "reason_codes", "archetype", "impressions_90d", "sessions_90d", "trend_direction", "age_tier",
]
main_queue = df[df["suggested_action"] != "technical_check_unranked"].sort_values(
    "model_probability", ascending=False
)
queue_path = WORK_OUTPUTS / "w07_action_playbook_queue.csv"
main_queue[export_cols].head(300).to_csv(queue_path, index=False)
print("Wrote:", queue_path, "-", min(300, len(main_queue)), "rows")

# 2. the separate technical-check list — full, since it is small
tech_path = WORK_OUTPUTS / "w07_unranked_technical_check.csv"
df[df["suggested_action"] == "technical_check_unranked"][export_cols].to_csv(tech_path, index=False)
print("Wrote:", tech_path, "-", (df["suggested_action"] == "technical_check_unranked").sum(), "rows")

# 3. metrics receipts — stays committed
summary_path = WORK_OUTPUTS / "w07_playbook_summary.json"
summary_path.write_text(json.dumps(monitoring_baseline, indent=2, sort_keys=True))
print("Wrote:", summary_path)

# 4. figures — self-contained SVG bar-chart helper, no external dependency
def simple_svg_bar_chart(title, labels, values, output_path, color="#426B69"):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    width = 900
    height = 520
    margin_left = 80
    margin_right = 40
    margin_top = 80
    margin_bottom = 110

    plot_width = width - margin_left - margin_right
    plot_height = height - margin_top - margin_bottom

    max_value = max(values) if values else 1
    bar_count = len(values)
    gap = 20

    bar_width = max(
        10,
        (plot_width - gap * (bar_count + 1)) / max(bar_count, 1)
    )

    svg = [
        f'<svg xmlns="http://www.w3.org/2000/svg" '
        f'width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        f'<text x="{width/2}" y="40" text-anchor="middle" '
        f'font-family="Arial, sans-serif" font-size="22" font-weight="bold">'
        f'{title}</text>',
    ]

    # Baseline
    baseline_y = margin_top + plot_height
    svg.append(
        f'<line x1="{margin_left}" y1="{baseline_y}" '
        f'x2="{width-margin_right}" y2="{baseline_y}" '
        f'stroke="#333" stroke-width="1"/>'
    )

    for i, (label, value) in enumerate(zip(labels, values)):
        x = margin_left + gap + i * (bar_width + gap)

        bar_height = (float(value) / max_value) * plot_height if max_value else 0
        y = baseline_y - bar_height

        svg.append(
            f'<rect x="{x:.2f}" y="{y:.2f}" '
            f'width="{bar_width:.2f}" height="{bar_height:.2f}" '
            f'fill="{color}"/>'
        )

        # Value label
        svg.append(
            f'<text x="{x + bar_width/2:.2f}" y="{y - 8:.2f}" '
            f'text-anchor="middle" font-family="Arial, sans-serif" '
            f'font-size="14">{int(value)}</text>'
        )

        # Category label
        safe_label = str(label).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        svg.append(
            f'<text x="{x + bar_width/2:.2f}" y="{baseline_y + 25}" '
            f'text-anchor="middle" font-family="Arial, sans-serif" '
            f'font-size="13" transform="rotate(-25 {x + bar_width/2:.2f} {baseline_y + 25})">'
            f'{safe_label}</text>'
        )

    svg.append('</svg>')

    output_path.write_text("\n".join(svg), encoding="utf-8")


simple_svg_bar_chart(
    "Suggested action mix",
    action_counts.index.tolist(),
    action_counts.values.tolist(),
    WORK_FIGURES / "action_mix.svg",
    color="#426B69",
)

conf_plot_counts = df["confidence"].value_counts().reindex(
    ["high", "medium", "low"],
    fill_value=0
)

simple_svg_bar_chart(
    "Playbook queue confidence",
    conf_plot_counts.index.tolist(),
    conf_plot_counts.values.tolist(),
    WORK_FIGURES / "confidence_mix.svg",
    color="#6F4E7C",
)

print("Wrote:", WORK_FIGURES / "action_mix.svg")
print("Wrote:", WORK_FIGURES / "confidence_mix.svg")

Wrote: ../outputs/w07_action_playbook_queue.csv - 300 rows
Wrote: ../outputs/w07_unranked_technical_check.csv - 1205 rows
Wrote: ../outputs/w07_playbook_summary.json
Wrote: ../figures/action_mix.svg
Wrote: ../figures/confidence_mix.svg


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.